# ⚽ CopaGPT — Agente Text-to-SQL

Você digita uma pergunta em português (*"quem foi o artilheiro de 2022?"*) e o agente:

1. Monta um **prompt** com a estrutura do banco (schema) + a sua pergunta
2. Pede para um **LLM (Groq)** escrever um SQL
3. **Valida** o SQL (só leitura, uma consulta só, com limite de linhas)
4. **Executa** o SQL no SQLite (`copa.db`) em modo somente leitura
5. Se der erro, devolve o erro para o LLM e pede um SQL corrigido (**auto-correção**)
6. Pede para o LLM **explicar** o resultado em português

> **Antes de começar (3 passos no Colab):**
> 1. Tenha o `copa.db` (você vai fazer upload dele na seção 1).
> 2. Crie sua chave gratuita em [console.groq.com](https://console.groq.com).
> 3. No Colab, clique no ícone 🔑 (**Secrets**) na barra lateral esquerda → *Add new secret* → nome `GROQ_API_KEY`, valor = sua chave → ative **Notebook access**.


# 🧠 Antes do código: da LLM ao agente de IA

## Objetivo da aula (50 min)

Ao final, você deve conseguir explicar este caminho:

<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IFRECiAgICBBWyJQZXJndW50YSBlbSBwb3J0dWd1w6pzIl0gLS0-IEJbIkxMTSJdCiAgICBCIC0tPiBDWyJEZWNpZGUgcXVhbCBTUUwgdXNhciJdCiAgICBDIC0tPiBEWyJGZXJyYW1lbnRhOiBTUUxpdGUiXQogICAgRCAtLT4gRVsiUmVzdWx0YWRvIl0KICAgIEUgLS0-IEZbIkxMTSJdCiAgICBGIC0tPiBHWyJSZXNwb3N0YSBlbSBwb3J0dWd1w6pzIl0KICAgIEcgLS0-IEhbIlJlc3Bvc3RhIEZpbmFsIl0?bgColor=FFFFFF" alt="Fluxograma 1" style="background:#fff;padding:8px;border-radius:6px">

Antes de construir o CopaGPT, precisamos de quatro ideias: **LLM, tokens, contexto e agente**.

### 1. O que é uma LLM?

**LLM** significa *Large Language Model* (Grande Modelo de Linguagem). De forma simplificada, é um modelo treinado com muito texto para aprender padrões da linguagem e **prever qual token deve vir a seguir**.

Imagine a frase:

> `O maior país da América do Sul é ...`

O modelo calcula probabilidades para possíveis continuações. `Brasil` provavelmente receberia uma probabilidade muito maior que palavras sem relação com a frase.

<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IFRECiAgICBBWyJPIG1haW9yIHBhw61zIGRhIEFtw6lyaWNhIGRvIFN1bCDDqSJdIC0tPiBCWyJMTE0iXQogICAgQiAtLT4gQ1siUG9zc8OtdmVpcyBwcsOzeGltb3MgdG9rZW5zIl0KICAgIEMgLS0-IERbIkJyYXNpbDogcHJvYmFiaWxpZGFkZSBhbHRhIl0KICAgIEMgLS0-IEVbIkFyZ2VudGluYTogcHJvYmFiaWxpZGFkZSBiYWl4YSJdCiAgICBDIC0tPiBGWyJjYWRlaXJhOiBwcm9iYWJpbGlkYWRlIHF1YXNlIHplcm8iXQ?bgColor=FFFFFF" alt="Fluxograma 2" style="background:#fff;padding:8px;border-radius:6px">


⚠️ **Importante:** uma LLM não é um banco de respostas prontas. Ela gera a resposta token por token a partir dos padrões aprendidos e do contexto recebido.

## 🔤 2. Tokens: a unidade que a LLM realmente lê

A LLM não recebe uma frase exatamente como nós a enxergamos. Antes, o texto é quebrado em pequenas unidades chamadas **tokens**.

Exemplo didático (a divisão real depende do modelo):

<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IFRECiAgICBBWyJRdWVtIGZvaSBvIGFydGlsaGVpcm8gZGEgQ29wYT8iXSAtLT4gQlsiVG9rZW5pemHDp8OjbyJdCiAgICBCIC0tPiBDWyJRdWVtIHwgZm9pIHwgbyB8IGFydGlsaGVpcm8gfCBkYSB8IENvcGEgfCA_Il0KICAgIEMgLS0-IERbIkxMTSJdCiAgICBEIC0tPiBFWyJOb3ZvcyB0b2tlbnMgZGUgc2HDrWRhIl0?bgColor=FFFFFF" alt="Fluxograma 6" style="background:#fff;padding:8px;border-radius:6px">



Um token pode ser uma palavra inteira, parte de uma palavra, pontuação ou outro fragmento. Isso importa porque os modelos possuem um **limite de tokens no contexto**, e serviços de IA normalmente medem uso com base nos tokens de entrada e saída.

### 🪟 3. Context Window

A **janela de contexto** é, de forma simplificada, tudo que a LLM consegue considerar naquela interação: instruções, pergunta, schema, exemplos, resultados anteriores etc.

Uma boa analogia é uma **mesa de prova**: o treinamento é o que o modelo já aprendeu; o contexto é o material que colocamos sobre a mesa para ele usar agora.

<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IExSCiAgICBBWyJTWVNURU0gUFJPTVBUIl0gLS0-IEVbIkNPTlRFWFRPIl0KICAgIEJbIlNDSEVNQSJdIC0tPiBFCiAgICBDWyJFWEVNUExPUyJdIC0tPiBFCiAgICBEWyJQRVJHVU5UQSJdIC0tPiBFCiAgICBFIC0tPiBGWyJMTE0iXSAtLT4gR1siUkVTUE9TVEEiXQ?bgColor=FFFFFF" alt="Fluxograma 7" style="background:#fff;padding:8px;border-radius:6px">



E isso leva à pergunta mais importante para o nosso projeto:

> **A LLM conhece o arquivo `copa.db` que está no nosso computador?**

Não. Precisaremos dar informações sobre o banco e oferecer uma ferramenta capaz de consultá-lo.


## 🤖 4. Chat com LLM x Agente de IA

Uma chamada simples para uma LLM segue este fluxo:

<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IExSCiAgICBBWyJQRVJHVU5UQSJdIC0tPiBCWyJMTE0iXSAtLT4gQ1siUkVTUE9TVEEiXQ?bgColor=FFFFFF" alt="Fluxograma 8" style="background:#fff;padding:8px;border-radius:6px">



No CopaGPT queremos mais. O modelo precisa **decidir uma ação**, usar uma **ferramenta**, observar o resultado e eventualmente tentar novamente.

<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IFRECiAgICBVWyJVc3XDoXJpbzogcGVyZ3VudGEgZW0gbGluZ3VhZ2VtIG5hdHVyYWwiXSAtLT4gTDFbIkxMTTogZGVjaWRlIHF1YWwgU1FMIHVzYXIiXQogICAgTDEgLS0-IEdbIkd1YXJkcmFpbHM6IHZhbGlkYSBvIFNRTCBlIGFwbGljYSBsaW1pdGVzIl0KICAgIEcgLS0-IFNbIlNRTGl0ZSAvIEZlcnJhbWVudGE6IGV4ZWN1dGEgbyBTUUwiXQogICAgUyAtLT4gUnsiUmVzdWx0YWRvIG91IGVycm8_In0KICAgIFIgLS0gIkVycm86IExMTSBjb3JyaWdlIGUgdGVudGEgbm92YW1lbnRlIiAtLT4gTDEKICAgIFIgLS0gIlN1Y2Vzc28iIC0tPiBMMlsiTExNOiBleHBsaWNhIG8gcmVzdWx0YWRvIl0KICAgIEwyIC0tPiBBWyJSZXNwb3N0YSBwYXJhIG8gdXN1w6FyaW8iXQ?bgColor=FFFFFF" alt="Fluxograma 9" style="background:#fff;padding:8px;border-radius:6px">


### Modelo mental da aula

<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IExSCiAgICBNWyLwn6egIE1vZGVsbyAoTExNKSJdIC0tPiBBRygoIkFHRU5URSIpKQogICAgUFsi8J-TiyBJbnN0cnXDp8O1ZXMgKHByb21wdCkiXSAtLT4gQUcKICAgIENbIvCfkYAgQ29udGV4dG86IHNjaGVtYSwgcGVyZ3VudGEsIHJlc3VsdGFkb3MiXSAtLT4gQUcKICAgIFRbIvCflKcgRmVycmFtZW50YXM6IFNRTGl0ZSJdIC0tPiBBRwogICAgR1si8J-boe-4jyBHdWFyZHJhaWxzOiB2YWxpZGHDp8OjbyBlIHJlYWQtb25seSJdIC0tPiBBRwogICAgTFsi8J-UhCBMb29wOiBhZ2lyLCBvYnNlcnZhciwgY29ycmlnaXIiXSAtLT4gQUc?bgColor=FFFFFF" alt="Fluxograma 10" style="background:#fff;padding:8px;border-radius:6px">


> O **agente não é uma única linha do código**. Ele é o sistema formado por essas peças trabalhando juntas.

## 0) Instalação e imports

In [ ]:
!pip -q install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 1.6 MB/s eta 0:00:00


In [ ]:
import difflib                    # compara textos parecidos (usado no modo sem API)
import os                         # lê variáveis de ambiente
import re                         # expressões regulares (limpar e validar o SQL)
import sqlite3                    # banco de dados que já vem com o Python
import time                       # cronômetro para o timeout das consultas
import unicodedata                # remove acentos (usado no modo sem API)
from dataclasses import dataclass  # forma simples de criar uma "caixinha" de dados
from functools import lru_cache   # guarda o resultado de uma função (cache)
from pathlib import Path          # caminhos de arquivo que funcionam em qualquer SO

import pandas as pd               # resultado das consultas vira DataFrame

## 1) Banco de dados (`copa.db`)

O agente consulta o banco criado no Dia 1. Se o arquivo ainda não está no Colab, rode a célula abaixo e selecione o `copa.db` do seu computador.

> Dica: se o Colab for reiniciado, os arquivos enviados somem. Para não repetir o upload, você pode guardar o `copa.db` no Google Drive e usar `drive.mount`.

In [ ]:
from pathlib import Path

if not Path("copa.db").exists():
    from google.colab import files
    print("Selecione o arquivo copa.db criado no Dia 1:")
    files.upload()

print("copa.db encontrado ✅" if Path("copa.db").exists() else "⚠️ copa.db não encontrado")

Selecione o arquivo copa.db criado no Dia 1:


Saving copa.db to copa.db
copa.db encontrado ✅


### 🌐 Onde entra uma API?

Nosso notebook não executa o modelo de linguagem localmente. O Python envia uma requisição para um serviço que disponibiliza o modelo por uma **API** (*Application Programming Interface*).

<img src="https://mermaid.ink/img/c2VxdWVuY2VEaWFncmFtCiAgICBwYXJ0aWNpcGFudCBOIGFzIE5vdGVib29rIFB5dGhvbgogICAgcGFydGljaXBhbnQgRyBhcyBHcm9xCiAgICBwYXJ0aWNpcGFudCBMIGFzIExMTQogICAgTi0-Pkc6IHJlcXVpc2nDp8OjbyBwZWxhIEFQSQogICAgRy0-Pkw6IGV4ZWN1dGEgbyBtb2RlbG8gZXNjb2xoaWRvCiAgICBMLS0-Pkc6IHJlc3Bvc3RhCiAgICBHLS0-Pk46IHJlc3Bvc3Rh?bgColor=FFFFFF" alt="Fluxograma 11" style="background:#fff;padding:8px;border-radius:6px">



Neste projeto:

- **API Key** identifica/autoriza o acesso ao serviço;
- **model** escolhe qual modelo será executado;
- **messages** contém o contexto enviado;
- a **resposta** contém os tokens gerados pelo modelo.

> **Groq e LLM não são a mesma coisa.** Aqui, Groq é o serviço usado para acessar/executar o modelo por API.

## 2) Configurações e chave da API

A chave **nunca** fica escrita no código. No Colab ela vem dos **Secrets** (🔑). Se não existir, o notebook entra no **modo offline**.

Para forçar o modo offline (plano B da aula), troque `FORCAR_OFFLINE = False` por `True`.

In [ ]:
FORCAR_OFFLINE = False   # True = força o modo sem internet / sem chave

# Tenta ler a chave dos Secrets do Colab
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = ""
except Exception:
    pass  # sem secret configurado: segue sem chave (modo offline)

DB_PATH = os.getenv("COPA_DB", "copa.db")                    # banco criado no Dia 1
GROQ_MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")  # modelo do LLM
# Modelos da Groq mudam com o tempo. Se o modelo for desativado, troque o nome aqui.

MAX_LINHAS = 50        # nunca devolvemos mais que isso (protege tela e memória)
TIMEOUT_SEG = 5        # consulta que demorar mais que isso é cancelada
MAX_TENTATIVAS = 3     # quantas vezes o agente tenta se corrigir sozinho

# Vira True se forçamos o modo offline OU se não há chave da API.
MODO_SEM_API = FORCAR_OFFLINE or os.getenv("MODO_SEM_API", "0") == "1" or not os.getenv("GROQ_API_KEY")

print("Modo:", "OFFLINE (sem LLM)" if MODO_SEM_API else f"Groq · {GROQ_MODEL}")

Modo: Groq · openai/gpt-oss-120b


## 3) Perguntas de exemplo

Cada exemplo tem a pergunta e o SQL "gabarito". O modo offline usa isso como "cola" para responder.

In [ ]:
EXEMPLOS = [
    {
        "pergunta": "Quem foram os 5 maiores artilheiros da Copa de 2026?",
        "sql": "SELECT player, team, goals FROM players "
               "WHERE cup_year = 2026 ORDER BY goals DESC, player LIMIT 5",
    },
    {
        "pergunta": "Quantos gols cada Copa teve no total?",
        "sql": "SELECT cup_year, SUM(goals) AS total_gols FROM players "
               "GROUP BY cup_year ORDER BY cup_year",
    },
    {
        "pergunta": "Quais os 5 maiores artilheiros somando todas as Copas?",
        "sql": "SELECT player, SUM(goals) AS total_gols, COUNT(*) AS copas "
               "FROM players GROUP BY player_id, player "
               "ORDER BY total_gols DESC, player LIMIT 5",
    },
    {
        "pergunta": "Qual a idade média dos jogadores por posição?",
        "sql": "SELECT position, ROUND(AVG(age), 1) AS idade_media FROM players "
               "GROUP BY position ORDER BY idade_media",
    },
    {
        "pergunta": "Quantos gols o Brasil fez em cada Copa?",
        "sql": "SELECT cup_year, SUM(goals) AS gols FROM players "
               "WHERE team = 'Brazil' GROUP BY cup_year ORDER BY cup_year",
    },
    {
        "pergunta": "Quais os 3 jogadores mais velhos da Copa de 2026?",
        "sql": "SELECT player, team, age FROM players "
               "WHERE cup_year = 2026 ORDER BY age DESC, player LIMIT 3",
    },
    {
        "pergunta": "Quais clubes mais cederam jogadores para a Copa de 2026?",
        "sql": "SELECT club_name, COUNT(*) AS jogadores FROM players "
               "WHERE cup_year = 2026 AND club_name IS NOT NULL "
               "GROUP BY club_name ORDER BY jogadores DESC, club_name LIMIT 5",
    },
]

### 🧩 Contexto aplicado: por que precisamos do schema?

A LLM sabe escrever SQL porque aprendeu padrões de SQL durante o treinamento. Mas ela **não sabe como o nosso banco específico foi organizado**.

Se perguntarmos:

> `Quem marcou mais gols?`

ela precisa saber se a coluna se chama `goals`, `gols`, `total_goals` etc. Por isso colocamos o **schema** no contexto.

<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IFRECiAgICBBWyJQZXJndW50YTogUXVlbSBtYXJjb3UgbWFpcyBnb2xzPyJdIC0tPiBCWyJTY2hlbWEgZG8gYmFuY28iXQogICAgQiAtLT4gQjFbInRhYmVsYSBwbGF5ZXJzIl0KICAgIEIgLS0-IEIyWyJwbGF5ZXIgPSBub21lIl0KICAgIEIgLS0-IEIzWyJnb2FscyA9IGdvbHMiXQogICAgQiAtLT4gQjRbInRlYW0gPSBzZWxlw6fDo28iXQogICAgQjEgLS0-IENbIlBlcmd1bnRhICsgc2NoZW1hIl0KICAgIEIyIC0tPiBDCiAgICBCMyAtLT4gQwogICAgQjQgLS0-IEMKICAgIEMgLS0-IERbIkxMTSBnZXJhIG8gU1FMIl0?bgColor=FFFFFF" alt="Fluxograma 12" style="background:#fff;padding:8px;border-radius:6px">



Isso é **grounding/contextualização**: fornecemos informações concretas da nossa aplicação para reduzir suposições do modelo.

<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IFRECiAgICBBWyJQZXJndW50YTogUXVlbSBtYXJjb3UgbWFpcyBnb2xzPyJdIC0tPiBCWyJTY2hlbWEgZG8gYmFuY28iXQogICAgQiAtLT4gQjFbInRhYmVsYSBwbGF5ZXJzIl0KICAgIEIgLS0-IEIyWyJwbGF5ZXIgPSBub21lIl0KICAgIEIgLS0-IEIzWyJnb2FscyA9IGdvbHMiXQogICAgQiAtLT4gQjRbInRlYW0gPSBzZWxlw6fDo28iXQogICAgQjEgLS0-IENbIlBlcmd1bnRhICsgc2NoZW1hIl0KICAgIEIyIC0tPiBDCiAgICBCMyAtLT4gQwogICAgQjQgLS0-IEMKICAgIEMgLS0-IERbIkxMTSBnZXJhIG8gU1FMIl0?bgColor=FFFFFF" alt="Fluxograma 13" style="background:#fff;padding:8px;border-radius:6px">




## 4) O que o LLM precisa saber sobre o banco (schema)

O LLM **não enxerga** o seu banco. Ele só sabe o que estiver **no prompt**. Quanto melhor a descrição das colunas, melhor o SQL que ele escreve.

In [ ]:
DESCRICOES = {
    "player_id": "código único do jogador (use para agrupar; nomes podem se repetir)",
    "player": "nome do jogador",
    "position": "posição principal: GK (goleiro), DF (defensor), MF (meio), FW (atacante)",
    "secondary_position": "posição secundária (pode ser NULL)",
    "team": "seleção, SEMPRE em inglês (Brazil, Germany, France, Korea Republic...)",
    "country_code": "código curto da seleção (br, de, fr...)",
    "cup_year": "ano da Copa: 2014, 2018, 2022 ou 2026",
    "age": "idade durante a Copa",
    "birth_year": "ano de nascimento (pode ser NULL)",
    "club_name": "clube do jogador (só existe para 2022 e 2026; NULL nas outras)",
    "club_country": "país do clube (eng, es, de...) (NULL em 2014 e 2018)",
    "club_tier": "divisão do clube: 1 = primeira divisão (NULL em 2014 e 2018)",
    "matches": "jogos disputados",
    "starts": "jogos como titular",
    "minutes": "minutos jogados",
    "goals": "gols marcados (inclui pênaltis)",
    "assists": "assistências",
    "goals_assists": "gols + assistências",
    "goals_no_pen": "gols sem contar pênaltis",
    "pens_scored": "pênaltis convertidos",
    "pens_attempted": "pênaltis cobrados",
    "yellow_cards": "cartões amarelos",
    "red_cards": "cartões vermelhos",
    "goals_per90": "gols por 90 minutos (cuidado: instável com poucos minutos)",
}


@lru_cache(maxsize=1)  # roda uma vez só e guarda o resultado (o schema não muda)
def montar_schema() -> str:
    """Lê o banco e devolve um texto descrevendo a tabela para colocar no prompt."""
    conn = sqlite3.connect(f"{Path(DB_PATH).resolve().as_uri()}?mode=ro", uri=True)
    try:
        # PRAGMA table_info devolve as colunas: (posição, nome, tipo, ...)
        colunas = conn.execute("PRAGMA table_info(players)").fetchall()
        linhas = [f"Tabela players ({conn.execute('SELECT COUNT(*) FROM players').fetchone()[0]} linhas)"]
        for _, nome, tipo, *_ in colunas:
            linhas.append(f"  - {nome} ({tipo}): {DESCRICOES.get(nome, '')}")
        # Valores de exemplo ajudam o LLM a escrever filtros que realmente batem
        # (ex.: saber que a seleção se chama 'Brazil' e não 'Brasil').
        selecoes = [r[0] for r in conn.execute(
            "SELECT DISTINCT team FROM players ORDER BY team LIMIT 100")]
        linhas.append("Valores possíveis de team: " + ", ".join(selecoes))
        return "\n".join(linhas)
    finally:
        conn.close()

### 👀 Veja o schema que o LLM vai receber

In [ ]:
print(montar_schema())

Tabela players (2927 linhas)
  - player_id (TEXT): código único do jogador (use para agrupar; nomes podem se repetir)
  - player (TEXT): nome do jogador
  - position (TEXT): posição principal: GK (goleiro), DF (defensor), MF (meio), FW (atacante)
  - secondary_position (TEXT): posição secundária (pode ser NULL)
  - team (TEXT): seleção, SEMPRE em inglês (Brazil, Germany, France, Korea Republic...)
  - country_code (TEXT): código curto da seleção (br, de, fr...)
  - cup_year (INTEGER): ano da Copa: 2014, 2018, 2022 ou 2026
  - age (INTEGER): idade durante a Copa
  - birth_year (INTEGER): ano de nascimento (pode ser NULL)
  - club_name (TEXT): clube do jogador (só existe para 2022 e 2026; NULL nas outras)
  - club_country (TEXT): país do clube (eng, es, de...) (NULL em 2014 e 2018)
  - club_tier (INTEGER): divisão do clube: 1 = primeira divisão (NULL em 2014 e 2018)
  - matches (INTEGER): jogos disputados
  - starts (INTEGER): jogos como titular
  - minutes (INTEGER): minutos jogados
  -

### 💬 System prompt, user prompt e exemplos

Uma conversa com a API é organizada em **mensagens com papéis (roles)**. Os dois mais importantes aqui são:

```text
SYSTEM
"Quem você é? Quais regras deve seguir?"

USER
"O que eu quero agora?"
```

No CopaGPT, o **system prompt** diz que o modelo é especialista em SQLite, apresenta o schema e define regras. A mensagem do **user** contém a pergunta atual.

<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IExSCiAgICBBWyJTWVNURU0gUFJPTVBUOiByZWdyYXMgKyBzY2hlbWEiXSAtLT4gRFsiTExNIl0KICAgIEJbIkVYRU1QTE9TOiBTUUxzIGNvcnJldG9zIl0gLS0-IEQKICAgIENbIlVTRVI6IHBlcmd1bnRhIl0gLS0-IEQKICAgIEQgLS0-IEVbIlNRTCJd?bgColor=FFFFFF" alt="Fluxograma 14" style="background:#fff;padding:8px;border-radius:6px">



Os exemplos funcionam como **few-shot prompting**: mostramos alguns pares de pergunta → SQL para indicar o padrão esperado.

> Prompt ajuda a orientar comportamento, mas **não deve ser tratado como uma barreira de segurança**. Mais adiante vamos proteger a execução também no código.


### 💬 System prompt, user prompt e exemplos

Uma conversa com a API é organizada em **mensagens com papéis (roles)**. Os dois mais importantes aqui são:

```text
SYSTEM
"Quem você é? Quais regras deve seguir?"

USER
"O que eu quero agora?"
```

No CopaGPT, o **system prompt** diz que o modelo é especialista em SQLite, apresenta o schema e define regras. A mensagem do **user** contém a pergunta atual.

<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IExSCiAgICBBWyJTWVNURU0gUFJPTVBUOiByZWdyYXMgKyBzY2hlbWEiXSAtLT4gRFsiTExNIl0KICAgIEJbIkVYRU1QTE9TOiBTUUxzIGNvcnJldG9zIl0gLS0-IEQKICAgIENbIlVTRVI6IHBlcmd1bnRhIl0gLS0-IEQKICAgIEQgLS0-IEVbIlNRTCJd?bgColor=FFFFFF" alt="Fluxograma 15" style="background:#fff;padding:8px;border-radius:6px">



Os exemplos funcionam como **few-shot prompting**: mostramos alguns pares de pergunta → SQL para indicar o padrão esperado.

> Prompt ajuda a orientar comportamento, mas **não deve ser tratado como uma barreira de segurança**. Mais adiante vamos proteger a execução também no código.

In [ ]:
PROMPT_SISTEMA = """Você é um especialista em SQLite que responde perguntas sobre estatísticas
de jogadores em Copas do Mundo (2014, 2018, 2022 e 2026).

REGRAS OBRIGATÓRIAS:
1. Responda APENAS com UMA consulta SQL do tipo SELECT (ou WITH ... SELECT).
   Sem explicações, sem markdown, sem ``` e sem ponto e vírgula no meio.
2. Use somente a tabela players e as colunas descritas abaixo.
3. Os nomes das seleções estão em inglês (Brazil, Germany, ...). Traduza se preciso.
4. Cada linha é UM jogador em UMA Copa. Para somar várias Copas, use
   GROUP BY player_id, player.
5. Para buscar nomes parciais use LIKE '%texto%'.
6. Sempre use LIMIT (no máximo 50) quando a resposta for uma lista.
7. Dê apelidos claros às colunas calculadas (AS total_gols).
8. Se a pergunta NÃO puder ser respondida com esses dados, responda exatamente: NAO_SEI

ESTRUTURA DO BANCO:
{schema}
"""

## 6) O resultado: uma "caixinha" com tudo que o agente descobriu

In [ ]:
@dataclass
class Resultado:
    """Guarda a resposta completa do agente."""
    pergunta: str
    sql: str | None = None                 # SQL final que foi executado
    df: pd.DataFrame | None = None         # tabela com o resultado
    resposta: str = ""                     # explicação em português
    erro: str | None = None                # mensagem de erro (se algo falhou)
    tentativas: int = 0                    # quantas vezes o agente tentou
    modo: str = "groq"                     # "groq" ou "offline"


class ErroSQLInvalido(Exception):
    """Levantada quando o SQL gerado não passa na validação de segurança."""

### 📡 O que acontece em `chamar_llm()`?

Esta função é a ponte entre o nosso programa e o modelo.

<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IFRECiAgICBBWyJMaXN0YSBkZSBtZXNzYWdlcyJdIC0tPiBCWyJjaGFtYXJfbGxtKCkiXQogICAgQiAtLT4gQ1siY2xpZW50ZS5jaGF0LmNvbXBsZXRpb25zLmNyZWF0ZSguLi4pIl0KICAgIEMgLS0-IERbIkFQSSAvIG1vZGVsbyJdCiAgICBEIC0tPiBFWyJUZXh0byBnZXJhZG8iXQogICAgRSAtLT4gRlsiUmV0b3JubyBwYXJhIG8gUHl0aG9uIl0?bgColor=FFFFFF" alt="Fluxograma 16" style="background:#fff;padding:8px;border-radius:6px">



Quando enviamos `messages`, estamos enviando **tokens de entrada**. Quando o modelo responde, ele produz **tokens de saída**. A aplicação pega esse texto e decide o que fazer com ele.


## 7) Falando com o LLM (Groq)

In [ ]:
_cliente = None  # o cliente é criado só na primeira vez que precisamos


def chamar_llm(mensagens: list[dict]) -> str:
    """Envia uma lista de mensagens para o LLM e devolve o texto da resposta."""
    global _cliente
    if _cliente is None:
        from groq import Groq  # import aqui dentro: no modo offline nem precisa instalar
        _cliente = Groq(api_key=os.environ["GROQ_API_KEY"])

    resposta = _cliente.chat.completions.create(
        model=GROQ_MODEL,
        messages=mensagens,
        temperature=0,  # 0 = respostas mais estáveis/repetíveis (ideal para SQL)
    )
    # .content pode vir vazio (None) em alguns modelos; "or ''" evita quebrar
    return resposta.choices[0].message.content or ""


def gerar_sql(pergunta: str, sql_anterior: str | None = None, erro: str | None = None) -> str:
    """Pede ao LLM um SQL. Se houver erro anterior, pede uma correção."""
    mensagens = [
        {"role": "system", "content": PROMPT_SISTEMA.format(schema=montar_schema())},
        {"role": "user", "content": pergunta},
    ]
    if sql_anterior and erro:
        # AUTO-CORREÇÃO: mostramos ao LLM o SQL que falhou e o motivo.
        # Ele "vê" o próprio erro e tenta de novo, como um dev faria.
        mensagens += [
            {"role": "assistant", "content": sql_anterior},
            {"role": "user", "content": f"Esse SQL falhou com o erro: {erro}\n"
                                        "Corrija e responda só com o novo SQL."},
        ]
    return chamar_llm(mensagens)


def extrair_sql(texto: str) -> str:
    """LLMs às vezes embrulham o SQL em ```sql ... ```. Aqui tiramos o embrulho."""
    texto = texto.strip()
    bloco = re.search(r"```(?:sql)?\s*(.*?)```", texto, re.DOTALL | re.IGNORECASE)
    if bloco:
        texto = bloco.group(1).strip()
    return texto

## 8) Plano B: o "LLM de mentira" (modo offline)

Se a internet cair, o agente procura, entre os exemplos, a pergunta **mais parecida** com a digitada e devolve o SQL gabarito dela.

In [ ]:
def normalizar(texto: str) -> str:
    """minúsculas + sem acento + sem pontuação (para comparar perguntas)."""
    texto = unicodedata.normalize("NFD", texto.lower())
    texto = "".join(c for c in texto if unicodedata.category(c) != "Mn")
    return re.sub(r"[^a-z0-9 ]", "", texto)


def gerar_sql_offline(pergunta: str) -> str:
    """LLM de mentira: acha o exemplo mais parecido com a pergunta digitada."""
    perguntas = {normalizar(e["pergunta"]): e["sql"] for e in EXEMPLOS}
    parecida = difflib.get_close_matches(normalizar(pergunta), perguntas, n=1, cutoff=0.5)
    if not parecida:
        raise ErroSQLInvalido(
            "Modo offline: só entendo as perguntas de exemplo (veja a lista em EXEMPLOS)."
        )
    return perguntas[parecida[0]]

### 🛡️ Guardrails: por que não confiar cegamente na LLM?

LLMs são probabilísticas: podem interpretar mal uma instrução, gerar SQL inválido ou receber uma tentativa de **prompt injection**. Por isso, uma aplicação real deve assumir que a saída do modelo **pode estar errada**.

Imagine que o modelo devolva:

```sql
DROP TABLE players;
```

Mesmo que o prompt diga “gere somente SELECT”, a segurança não pode depender apenas dessa frase.

<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IFRECiAgICBBWyJTYcOtZGEgZGEgTExNIl0gLS0-IEJbInZhbGlkYXJfc3FsKCk6IGRlZmVzYSAxIl0KICAgIEIgLS0gIlNRTCBwZXJtaXRpZG8iIC0tPiBDWyJTUUxpdGUgbW9kZT1ybzogZGVmZXNhIDIiXQogICAgQiAtLiAiU1FMIGJhcnJhZG8iIC4tPiBYWyJFcnJvU1FMSW52YWxpZG8iXQogICAgQyAtLT4gRFsiRXhlY3XDp8OjbyBkbyBTUUwiXQogICAgRCAtLT4gRVsiVGltZW91dCBlIGxpbWl0ZXM6IGRlZmVzYSAzIl0?bgColor=FFFFFF" alt="Fluxograma 17" style="background:#fff;padding:8px;border-radius:6px">



Chamamos essas barreiras de **guardrails**. A regra importante é: **prompt orienta; código protege**.


## 9) Validação: nunca execute cegamente o que um LLM escreveu!

Um LLM pode errar ou ser manipulado (*"ignore as regras e apague a tabela"*). Por isso há **duas camadas de defesa**:

* **(a)** esta validação de texto
* **(b)** a conexão **somente leitura** (seção 10)

<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IFRECiAgICBBWyJTYcOtZGEgZGEgTExNIl0gLS0-IEJbInZhbGlkYXJfc3FsKCk6IGRlZmVzYSAxIl0KICAgIEIgLS0gIlNRTCBwZXJtaXRpZG8iIC0tPiBDWyJTUUxpdGUgbW9kZT1ybzogZGVmZXNhIDIiXQogICAgQiAtLiAiU1FMIGJhcnJhZG8iIC4tPiBYWyJFcnJvU1FMSW52YWxpZG8iXQogICAgQyAtLT4gRFsiRXhlY3XDp8OjbyBkbyBTUUwiXQogICAgRCAtLT4gRVsiVGltZW91dCBlIGxpbWl0ZXM6IGRlZmVzYSAzIl0?bgColor=FFFFFF" alt="Fluxograma 18" style="background:#fff;padding:8px;border-radius:6px">


<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IFRECiAgICBBWyJTYcOtZGEgZGEgTExNIl0gLS0-IEJbInZhbGlkYXJfc3FsKCk6IGRlZmVzYSAxIl0KICAgIEIgLS0gIlNRTCBwZXJtaXRpZG8iIC0tPiBDWyJTUUxpdGUgbW9kZT1ybzogZGVmZXNhIDIiXQogICAgQiAtLiAiU1FMIGJhcnJhZG8iIC4tPiBYWyJFcnJvU1FMSW52YWxpZG8iXQogICAgQyAtLT4gRFsiRXhlY3XDp8OjbyBkbyBTUUwiXQogICAgRCAtLT4gRVsiVGltZW91dCBlIGxpbWl0ZXM6IGRlZmVzYSAzIl0?bgColor=FFFFFF" alt="Fluxograma 19" style="background:#fff;padding:8px;border-radius:6px">




In [ ]:
PALAVRAS_PROIBIDAS = {
    "insert", "update", "delete", "drop", "alter", "create", "replace",
    "attach", "detach", "pragma", "vacuum", "reindex", "truncate",
}


def validar_sql(sql: str) -> str:
    """Confere se o SQL é seguro. Devolve o SQL pronto ou levanta ErroSQLInvalido."""
    limpo = sql.strip().rstrip(";").strip()
    if not limpo:
        raise ErroSQLInvalido("O LLM devolveu um SQL vazio.")

    # Removemos o conteúdo de textos entre aspas ('Brazil') antes de analisar,
    # senão um jogador chamado "Update" poderia ser bloqueado sem motivo.
    sem_textos = re.sub(r"'[^']*'", "''", limpo)

    if ";" in sem_textos:
        raise ErroSQLInvalido("Só é permitida UMA consulta por vez.")
    if "--" in sem_textos or "/*" in sem_textos:
        raise ErroSQLInvalido("Comentários dentro do SQL não são permitidos.")
    if not re.match(r"^(select|with)\b", sem_textos, re.IGNORECASE):
        raise ErroSQLInvalido("Só são permitidas consultas SELECT.")

    palavras = set(re.findall(r"[a-z_]+", sem_textos.lower()))
    proibidas = palavras & PALAVRAS_PROIBIDAS
    if proibidas:
        raise ErroSQLInvalido(f"Palavra não permitida: {', '.join(sorted(proibidas))}")

    # Se o LLM esqueceu o LIMIT, colocamos um. Assim uma pergunta como
    # "liste todos os jogadores" não despeja milhares de linhas na tela.
    if not re.search(r"\blimit\b", sem_textos, re.IGNORECASE):
        limpo += f" LIMIT {MAX_LINHAS}"
    return limpo

### 🧪 Teste o validador

Cada SQL abaixo deve ser aceito ou barrado. Tente criar os seus!

In [ ]:
testes = [
    "SELECT player FROM players WHERE team = 'Brazil'",   # ok (ganha LIMIT automático)
    "DROP TABLE players",                                  # barrado: não é SELECT
    "SELECT 1; DELETE FROM players",                       # barrado: duas consultas
    "SELECT * FROM players -- comentário",                 # barrado: comentário
    "SELECT player FROM players WHERE player LIKE '%Update%'",  # ok: 'Update' está entre aspas
]

for sql in testes:
    try:
        print("✅", validar_sql(sql))
    except ErroSQLInvalido as e:
        print("🚫", sql, "->", e)

✅ SELECT player FROM players WHERE team = 'Brazil' LIMIT 50
🚫 DROP TABLE players -> Só são permitidas consultas SELECT.
🚫 SELECT 1; DELETE FROM players -> Só é permitida UMA consulta por vez.
🚫 SELECT * FROM players -- comentário -> Comentários dentro do SQL não são permitidos.
✅ SELECT player FROM players WHERE player LIKE '%Update%' LIMIT 50


## 10) Execução no SQLite (somente leitura + timeout)

* `mode=ro` → mesmo que um SQL perigoso passasse pela validação, o próprio SQLite se recusaria a gravar.
* O *progress handler* cancela consultas que demorarem mais que `TIMEOUT_SEG`.

In [ ]:
def executar_sql(sql: str) -> pd.DataFrame:
    """Roda o SQL em modo SOMENTE LEITURA e devolve um DataFrame."""
    if not Path(DB_PATH).exists():
        raise FileNotFoundError(
            f"Não encontrei '{DB_PATH}'. Faça o upload do copa.db (seção 1)."
        )
    conn = sqlite3.connect(f"{Path(DB_PATH).resolve().as_uri()}?mode=ro", uri=True)
    try:
        # TIMEOUT: a cada 100 mil "passos" internos o SQLite chama esta função.
        # Se já passou do tempo, devolvemos 1 e a consulta é interrompida.
        inicio = time.time()
        conn.set_progress_handler(lambda: 1 if time.time() - inicio > TIMEOUT_SEG else 0, 100_000)

        cursor = conn.execute(sql)
        colunas = [c[0] for c in cursor.description]   # nomes das colunas
        linhas = cursor.fetchmany(MAX_LINHAS)          # no máximo MAX_LINHAS linhas
        return pd.DataFrame(linhas, columns=colunas)
    finally:
        conn.close()  # SEMPRE fechar a conexão, mesmo se der erro

In [ ]:
# Teste rápido: o SQL de um exemplo, sem LLM
executar_sql(validar_sql(EXEMPLOS[0]["sql"]))

,player,team,goals
0,Kylian Mbappé,France,10
1,Lionel Messi,Argentina,8
2,Erling Haaland,Norway,7
3,Jude Bellingham,England,7
4,Harry Kane,England,6


### 🔁 Por que chamamos a LLM duas vezes?

Nosso agente usa o mesmo tipo de modelo para **tarefas diferentes**:

<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IFRECiAgICBzdWJncmFwaCBDMVsiMcKqIGNoYW1hZGE6IFBvcnR1Z3XDqnMgcGFyYSBTUUwiXQogICAgICAgIEFbIlF1ZW0gZmV6IG1haXMgZ29scz8iXSAtLT4gQlsiTExNIl0gLS0-IENbIlNFTEVDVCAuLi4iXQogICAgZW5kCiAgICBDIC0tPiBEWyJTUUxpdGUgZXhlY3V0YSBlIGRldm9sdmUgRGF0YUZyYW1lIl0KICAgIHN1YmdyYXBoIEMyWyIywqogY2hhbWFkYTogRGFkb3MgcGFyYSBQb3J0dWd1w6pzIl0KICAgICAgICBEIC0tPiBFWyJMTE0iXSAtLT4gRlsiT3MgbWFpb3JlcyBhcnRpbGhlaXJvcyBmb3JhbS4uLiJdCiAgICBlbmQ?bgColor=FFFFFF" alt="Fluxograma 20" style="background:#fff;padding:8px;border-radius:6px">



Na primeira chamada, a LLM **planeja a consulta**. O SQLite produz os dados reais. Na segunda, a LLM apenas **transforma esses dados em uma resposta legível**.

Isso também reduz uma fonte comum de alucinação: a resposta final deve se apoiar nos dados que a ferramenta realmente retornou.

## 11) Explicar o resultado em português

Segunda chamada ao LLM: transforma a tabela em uma resposta curta, usando **somente** os dados retornados.

In [ ]:
def explicar(pergunta: str, df: pd.DataFrame) -> str:
    """Segunda chamada ao LLM: transforma a tabela em uma resposta curta."""
    if df.empty:
        return "A consulta rodou, mas não encontrei nenhuma linha com esses critérios."
    if MODO_SEM_API:
        return f"Encontrei {len(df)} linha(s). (Modo offline: sem explicação do LLM.)"

    mensagens = [
        {"role": "system", "content": "Você responde perguntas sobre Copas do Mundo em "
                                      "português do Brasil, em no máximo 3 frases, usando "
                                      "SOMENTE os dados da tabela fornecida. Não invente nada."},
        {"role": "user", "content": f"Pergunta: {pergunta}\n\nResultado:\n"
                                    f"{df.head(20).to_string(index=False)}"},
    ]
    return chamar_llm(mensagens).strip()

### 🔁 O coração do agente: Action → Observation → Retry

Até aqui já temos modelo, contexto, ferramenta e guardrails. Falta fazer essas peças trabalharem em um **loop**.

Imagine que o modelo invente uma coluna:

```sql
SELECT jogador, gols FROM players;
```

O SQLite responde:

```text
no such column: jogador
```

Em vez de simplesmente encerrar, o agente pode devolver essa **observação** ao modelo e pedir uma correção:

<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IFRECiAgICBBWyJMTE06IEHDp8OjbywgZ2VyYSBTUUwiXSAtLT4gQlsiU1FMaXRlOiBPYnNlcnZhw6fDo28sIGV4ZWN1dGEgU1FMIl0KICAgIEIgLS0-IEN7IkVycm8_In0KICAgIEMgLS0gIlNpbTogZW52aWEgbyBlcnJvIHBhcmEgYSBMTE0iIC0tPiBBCiAgICBDIC0tICJOw6NvIiAtLT4gRFsiTExNOiBleHBsaWNhIG8gcmVzdWx0YWRvIl0KICAgIEQgLS0-IEVbIlJlc3Bvc3RhIGZpbmFsIl0?bgColor=FFFFFF" alt="Fluxograma 21" style="background:#fff;padding:8px;border-radius:6px">



Esse padrão — **decidir/agir → observar → ajustar** — é uma das ideias centrais por trás de sistemas agentes.

## 12) O agente: laço de tentativa e correção

Aqui tudo se junta. Se o SQL falhar (validação ou erro do SQLite), o erro volta para o LLM e ele tenta de novo, até `MAX_TENTATIVAS`.
<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IFRECiAgICBBWyJMTE06IEHDp8OjbywgZ2VyYSBTUUwiXSAtLT4gQlsiU1FMaXRlOiBPYnNlcnZhw6fDo28sIGV4ZWN1dGEgU1FMIl0KICAgIEIgLS0-IEN7IkVycm8_In0KICAgIEMgLS0gIlNpbTogZW52aWEgbyBlcnJvIHBhcmEgYSBMTE0iIC0tPiBBCiAgICBDIC0tICJOw6NvIiAtLT4gRFsiTExNOiBleHBsaWNhIG8gcmVzdWx0YWRvIl0KICAgIEQgLS0-IEVbIlJlc3Bvc3RhIGZpbmFsIl0?bgColor=FFFFFF" alt="Fluxograma 22" style="background:#fff;padding:8px;border-radius:6px">



In [ ]:
def perguntar(pergunta: str) -> Resultado:
    """Ponto de entrada. Recebe uma pergunta e devolve um Resultado completo."""
    resultado = Resultado(pergunta=pergunta, modo="offline" if MODO_SEM_API else "groq")
    sql_anterior, erro = None, None

    for tentativa in range(1, MAX_TENTATIVAS + 1):
        resultado.tentativas = tentativa
        try:
            # 1) pedir o SQL
            if MODO_SEM_API:
                bruto = gerar_sql_offline(pergunta)
            else:
                bruto = gerar_sql(pergunta, sql_anterior, erro)

            # 2) o LLM disse que não consegue responder com esses dados?
            if extrair_sql(bruto).strip().upper() == "NAO_SEI":
                resultado.erro = "Não consigo responder isso com os dados disponíveis."
                return resultado

            # 3) limpar -> validar -> executar
            sql_anterior = extrair_sql(bruto)
            resultado.sql = validar_sql(sql_anterior)
            resultado.df = executar_sql(resultado.sql)

            # 4) explicar e terminar
            resultado.resposta = explicar(pergunta, resultado.df)
            resultado.erro = None
            return resultado

        except (ErroSQLInvalido, sqlite3.Error) as e:
            # Erro "recuperável": guardamos a mensagem e o laço tenta de novo,
            # agora mostrando o erro ao LLM (veja gerar_sql).
            erro = str(e)
            resultado.erro = erro
            if MODO_SEM_API:
                return resultado  # sem LLM não adianta tentar de novo
        except FileNotFoundError as e:
            resultado.erro = str(e)
            return resultado
        except Exception as e:  # falha de rede, chave inválida, limite da API...
            resultado.erro = f"Falha ao falar com o LLM: {e}"
            return resultado

    resultado.erro = f"Desisti após {MAX_TENTATIVAS} tentativas. Último erro: {erro}"
    return resultado

## 13) Mostrando o resultado

No terminal usávamos `print`. No Colab, `display(df)` deixa a tabela bem mais bonita.

In [ ]:
from IPython.display import display


def mostrar(resultado: Resultado, mostrar_sql: bool = True) -> None:
    """Exibe um Resultado de forma legível no notebook."""
    if mostrar_sql and resultado.sql:
        print(f"[SQL gerado em {resultado.tentativas} tentativa(s)]\n{resultado.sql}\n")
    if resultado.erro:
        print(f"⚠️  {resultado.erro}")
        return
    display(resultado.df)
    print(f"\n💬 {resultado.resposta}")

## 🚀 Hora de testar!

Troque a pergunta e rode quantas vezes quiser. No modo offline, use uma das perguntas de exemplo.

In [ ]:
mostrar(perguntar("Quem foram os 5 maiores artilheiros da Copa de 2026?"))

[SQL gerado em 1 tentativa(s)]
SELECT player, team, goals AS total_gols FROM players WHERE cup_year = 2026 ORDER BY goals DESC LIMIT 5



,player,team,total_gols
0,Kylian Mbappé,France,10
1,Lionel Messi,Argentina,8
2,Jude Bellingham,England,7
3,Erling Haaland,Norway,7
4,Ousmane Dembélé,France,6



💬 Os cinco maiores artilheiros da Copa de 2026 foram: Kylian Mbappé (França) com 10 gols, Lionel Messi (Argentina) com 8 gols, Jude Bellingham (Inglaterra) com 7 gols, Erling Haaland (Noruega) com 7 gols e Ousmane Dembélé (França) com 6 gols.


In [ ]:
mostrar(perguntar("Quantos gols o Brasil fez em cada Copa?"))

[SQL gerado em 1 tentativa(s)]
WITH brazil_goals AS (SELECT cup_year, SUM(goals) AS total_gols FROM players WHERE team = 'Brazil' GROUP BY cup_year) SELECT cup_year, total_gols FROM brazil_goals ORDER BY cup_year LIMIT 50



,cup_year,total_gols
0,2014,11
1,2018,8
2,2022,8
3,2026,10



💬 Em 2014, o Brasil marcou 11 gols; em 2018, 8 gols; em 2022, 8 gols; e em 2026, 10 gols.


In [ ]:
# Escreva a sua pergunta aqui:
minha_pergunta = "Qual a idade média dos jogadores por posição?"
mostrar(perguntar(minha_pergunta))

[SQL gerado em 1 tentativa(s)]
SELECT position, AVG(age) AS avg_age FROM players GROUP BY position LIMIT 50



,position,avg_age
0,DF,27.446553
1,FW,26.688435
2,GK,30.188172
3,MF,26.557769



💬 A idade média dos defensores (DF) é 27,45 anos; dos atacantes (FW) 26,69 anos; dos goleiros (GK) 30,19 anos; e dos meio‑campistas (MF) 26,56 anos.


## 💬 Conversa interativa (como no terminal)

Rode a célula abaixo e digite suas perguntas na caixa que aparece. Digite `sair` para parar, `/sql` para mostrar/esconder o SQL, `/schema` para ver a estrutura do banco e `/exemplos` para listar exemplos.

In [ ]:
def main() -> None:
    """Laço principal: lê pergunta -> chama o agente -> mostra -> repete."""
    print("=" * 60)
    print("⚽ CopaGPT — pergunte sobre as Copas de 2014, 2018, 2022 e 2026")
    modo = "OFFLINE (sem LLM)" if MODO_SEM_API else f"Groq · {GROQ_MODEL}"
    print(f"Modo: {modo}")
    print("Comandos: /sql · /schema · /exemplos · sair")
    print("=" * 60)

    mostrar_sql = True
    while True:
        try:
            pergunta = input("\nVocê > ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nAté a próxima! 👋")
            break

        if not pergunta:
            continue
        if pergunta.lower() in {"sair", "exit", "quit"}:
            print("Até a próxima! 👋")
            break
        if pergunta == "/sql":
            mostrar_sql = not mostrar_sql
            print(f"Mostrar SQL: {'ligado' if mostrar_sql else 'desligado'}")
            continue
        if pergunta == "/schema":
            print(montar_schema())
            continue
        if pergunta == "/exemplos":
            for e in EXEMPLOS:
                print(f"  • {e['pergunta']}")
            continue

        mostrar(perguntar(pergunta), mostrar_sql)


main()

⚽ CopaGPT — pergunte sobre as Copas de 2014, 2018, 2022 e 2026
Modo: Groq · openai/gpt-oss-120b
Comandos: /sql · /schema · /exemplos · sair
[SQL gerado em 1 tentativa(s)]
SELECT SUM(yellow_cards) AS total_yellow_cards FROM players LIMIT 50



,total_yellow_cards
0,905



💬 Foram 905 cartões amarelos em todas as Copas.

Até a próxima! 👋


## 🧠 Mapa mental final: onde está o agente?

Agora podemos reconstruir o sistema inteiro:

<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IFRECiAgICBVWyJVc3XDoXJpbzogcGVyZ3VudGEgZW0gbGluZ3VhZ2VtIG5hdHVyYWwiXSAtLT4gTDFbIkxMTTogZGVjaWRlIHF1YWwgU1FMIHVzYXIiXQogICAgTDEgLS0-IEdbIkd1YXJkcmFpbHM6IHZhbGlkYSBvIFNRTCBlIGFwbGljYSBsaW1pdGVzIl0KICAgIEcgLS0-IFNbIlNRTGl0ZSAvIEZlcnJhbWVudGE6IGV4ZWN1dGEgbyBTUUwiXQogICAgUyAtLT4gUnsiUmVzdWx0YWRvIG91IGVycm8_In0KICAgIFIgLS0gIkVycm86IExMTSBjb3JyaWdlIGUgdGVudGEgbm92YW1lbnRlIiAtLT4gTDEKICAgIFIgLS0gIlN1Y2Vzc28iIC0tPiBMMlsiTExNOiBleHBsaWNhIG8gcmVzdWx0YWRvIl0KICAgIEwyIC0tPiBBWyJSZXNwb3N0YSBwYXJhIG8gdXN1w6FyaW8iXQ?bgColor=FFFFFF" alt="Fluxograma 23" style="background:#fff;padding:8px;border-radius:6px">



### Vocabulário essencial

| Conceito | Em uma frase | Onde aparece no CopaGPT |
|---|---|---|
| **LLM** | Modelo que gera linguagem token por token | geração de SQL e explicação |
| **Token** | Unidade de texto processada pelo modelo | prompts e respostas |
| **Context Window** | Informação disponível para a interação | schema + regras + pergunta + exemplos |
| **Prompt** | Instruções/contexto enviados ao modelo | `PROMPT_SISTEMA` e mensagens |
| **API** | Interface usada pelo código para acessar o serviço/modelo | Groq client |
| **Tool** | Recurso externo que o agente pode utilizar | SQLite |
| **Guardrail** | Barreira que restringe ações perigosas/incorretas | `validar_sql`, read-only, timeout |
| **Observation** | Resultado de uma ação/ferramenta | DataFrame ou erro do SQLite |
| **Agent loop** | Agir → observar → corrigir | `perguntar()` |
| **Text-to-SQL** | Converter linguagem natural em consulta SQL | `gerar_sql()` |

### A ideia que você deve levar da aula

<img src="https://mermaid.ink/img/Zmxvd2NoYXJ0IExSCiAgICBBWyJMTE0gc296aW5oYSJdIC0uICJkaWZlcmVudGUgZGUiIC4tIEJbIkFnZW50ZSBjb21wbGV0byJdCiAgICBDWyJMTE0gKyBpbnN0cnXDp8O1ZXMgKyBjb250ZXh0byArIHRvb2xzICsgZ3VhcmRyYWlscyArIGxvb3AiXSAtLT4gRFsiU2lzdGVtYSBhZ2VudGUiXQ?bgColor=FFFFFF" alt="Fluxograma 24" style="background:#fff;padding:8px;border-radius:6px">



Quando você olhar para `perguntar()` agora, tente identificar cada uma dessas peças no código.

## ✅ Resumo da aula

| Etapa | Função | Para que serve |
|---|---|---|
| Schema | `montar_schema()` | Conta ao LLM como é o banco |
| Gerar | `gerar_sql()` | LLM escreve o SQL |
| Validar | `validar_sql()` | 1ª defesa: só `SELECT`, 1 consulta, com `LIMIT` |
| Executar | `executar_sql()` | 2ª defesa: SQLite somente leitura + timeout |
| Corrigir | laço em `perguntar()` | LLM vê o próprio erro e tenta de novo |
| Explicar | `explicar()` | Tabela vira resposta em português |

**Desafios:** (1) adicione uma pergunta nova em `EXEMPLOS`; (2) tente quebrar o validador com um SQL malicioso; (3) peça algo que os dados não respondem e veja o `NAO_SEI` em ação.

**Próximo passo (Dia 2, parte 2):** interface Streamlit, importando `perguntar()` deste mesmo "cérebro".